<a href="https://colab.research.google.com/github/farithjhg/aws-bedrock-agentcore-course/blob/main/v7_aprender_haciendo/labs/09_identidad.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

🎬 [La escena en la presentación](https://farithjhg.github.io/aws-bedrock-agentcore-course/v7_aprender_haciendo/presentacion_nova_assist.html#cap09) · 🧭 [Teoría del capítulo](https://farithjhg.github.io/aws-bedrock-agentcore-course/v7_aprender_haciendo/labs/teoria/cap09-identidad.html) · 🗺️ [El mapa](https://farithjhg.github.io/aws-bedrock-agentcore-course/v7_aprender_haciendo/labs/teoria/index.html) · ⬅️ [Cap. 8](https://colab.research.google.com/github/farithjhg/aws-bedrock-agentcore-course/blob/main/v7_aprender_haciendo/labs/08_rag_iceberg_s3vectors.ipynb) · ➡️ [Cap. 10](https://colab.research.google.com/github/farithjhg/aws-bedrock-agentcore-course/blob/main/v7_aprender_haciendo/labs/10_policy_guardrails.ipynb)

# Capítulo 9 — Saber quién habla

**Formación Amazon Bedrock AgentCore · octubre 2026 · Caso de uso: Nova Assist (TiendaNova)**

> 🎬 **Escena.** Alguien escribe con el `cliente_id` de Laura y pregunta *«¿cómo preferís contactarme?»*. El agente le cuenta sus preferencias.

| | |
|---|---|
| **🧠 Componente agéntico** | Identidad y delegación |
| **La necesidad** | Que la identidad del cliente esté **verificada**: que no la escriba nadie en el payload ni la deduzca el modelo |
| **Lo que construimos** | JWT *inbound* en el Runtime (Cognito), `actor_id` desde el token y patrón OBO |
| **🏠 En local** | ☁️ No aplica: en tu portátil no hay clientes que identificar. |
| **☁️ Por qué AWS al salir a Internet** | En local, los secretos viven en un `.env` y los tokens se gestionan a mano. AgentCore Identity valida tokens, guarda credenciales en un Token Vault cifrado y permite actuar en nombre del usuario (OBO) sin superar sus permisos. |
| **En el ordenador sería…** | Las **cuentas de usuario** y el inicio de sesión del sistema operativo |
| **Requisitos del caso** | RNF-01, RNF-02 |
| **Prerrequisito · duración** | Capítulo 8 · 60 min |
| **⚠️ Dónde se rompe la analogía** | Iniciar sesión en un ordenador identifica a la persona; aquí hay **dos identidades**, la del agente y la del cliente, y confundirlas abre la puerta a los accesos indebidos. |

### ☁️ ¿Estás en Google Colab? Ejecuta primero esta celda

En tu equipo no hace nada. En Colab descarga el curso, monta tu **Google Drive** (ahí se guarda tu estado entre capítulos: `MyDrive/NovaAssist_curso`), instala las dependencias y la AgentCore CLI 0.31.1 y carga tus credenciales de AWS desde los **Secretos** de Colab. Tarda 2–3 minutos, porque cada notebook abre una máquina nueva: mientras tanto, lee la escena.

In [ ]:
# ☁️ Solo en Google Colab (en tu equipo no hace nada)
import os, sys, subprocess
if "google.colab" in sys.modules:
    CURSO = "/content/aws-bedrock-agentcore-course"
    if not os.path.isdir(CURSO):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", "main",
                        "https://github.com/farithjhg/aws-bedrock-agentcore-course.git", CURSO], check=True)
    os.chdir(f"{CURSO}/v7_aprender_haciendo/labs")
    sys.path.insert(0, os.getcwd())
    import colab_nova
    colab_nova.preparar(capitulo=9)

In [ ]:
# Configuración común: ejecuta siempre esta celda primero
import os
from lab_utils import *

os.environ.setdefault("AWS_REGION", region())
cuaderno()

> 🎮 **¿Llegas tarde o algo se rompió?** La celda siguiente comprueba que tienes todo lo necesario. Si no, descomenta `ponerse_al_dia(9)`: carga la partida guardada al final del capítulo anterior o, si no existe, escribe el código de referencia y te dice qué recursos faltan y en qué capítulo se crean.

In [ ]:
requisitos(9)
# ponerse_al_dia(9)   # ← descomenta si llegas tarde o algo se rompió

## 0. Experimento de rotura: suplantar a Laura

Hasta hoy, el agente sabe quién es el cliente porque alguien lo escribe en el payload. Un desconocido escribe el `cliente_id` de Laura.

> ✋ **Predice antes de ejecutar.** ¿Responderá con las preferencias de Laura a cualquiera que ponga `cliente_id="C-001"` en el payload? ¿Qué lo impide hoy?

In [ ]:
# Un desconocido escribe el cliente_id de Laura en el payload (capítulos 4-8: la identidad era un campo libre)
invocar("¿Cómo preferís contactarme si hay un problema con mi envío?", cliente_id="C-001")
print("¿Ha revelado el canal preferido de Laura?", "whatsapp" in (ultima_respuesta() or "").lower())

In [ ]:
# Comprobación determinista: lo que el agente recibe de la memoria para quien dice ser C-001
buscar_en_memoria("C-001", "¿cómo prefiere que le contactemos?")

La memoria le entrega las preferencias de Laura a **cualquiera que escriba su identificador**: lo normal es que el agente responda «por WhatsApp» a un desconocido. Con la identidad en el payload, basta con escribir el identificador de otra persona.

> 🧭 **¿Por qué ha pasado?** Abre la teoría: [**Las cuentas de usuario del agente**](https://farithjhg.github.io/aws-bedrock-agentcore-course/v7_aprender_haciendo/labs/teoria/cap09-identidad.html). Elige quién escribe y con qué credencial, con y sin autorizador JWT en el Runtime. Vuelve cuando sepas responder a esta pregunta: **¿quién debe aportar la identidad del cliente, por qué no puede ser ni el payload ni el modelo, y qué dos identidades distintas viajan en cada petición?**

**En dos líneas:** en un sistema agéntico hay **dos identidades**: la del agente (qué software llama) y la del usuario (en nombre de quién). Autenticación es quién eres; autorización, qué puedes hacer; delegación (OBO), actuar con los permisos del usuario y nunca con más. **Regla de oro:** la identidad no la escribe ni el modelo ni el usuario en un campo libre; la aporta la plataforma. Este componente no existe en local: en tu portátil no hay clientes que identificar.

## 1. Pool de clientes (Cognito) y usuarios de prueba

In [ ]:
import boto3, secrets

idp = boto3.client("cognito-idp", region_name=region())
if state_get("CLIENTES_POOL_ID"):
    print("El pool de clientes ya existe")
else:
    pool = idp.create_user_pool(PoolName="nova-clientes")["UserPool"]["Id"]
    client_id = idp.create_user_pool_client(
        UserPoolId=pool, ClientName="nova-web",
        ExplicitAuthFlows=["ALLOW_USER_PASSWORD_AUTH", "ALLOW_REFRESH_TOKEN_AUTH"])["UserPoolClient"]["ClientId"]
    password = "Nova-" + secrets.token_urlsafe(8) + "1a!"
    for user in ("C-001", "C-002"):
        idp.admin_create_user(UserPoolId=pool, Username=user, MessageAction="SUPPRESS")
        idp.admin_set_user_password(UserPoolId=pool, Username=user, Password=password, Permanent=True)
    state_set(CLIENTES_POOL_ID=pool, CLIENTES_CLIENT_ID=client_id, CLIENTES_PASSWORD=password,
              CLIENTES_DISCOVERY_URL=f"https://cognito-idp.{region()}.amazonaws.com/{pool}/.well-known/openid-configuration")

In [ ]:
import base64, json

tok = token_cliente("C-001")
claims = json.loads(base64.urlsafe_b64decode(tok.split(".")[1] + "=" * (-len(tok.split(".")[1]) % 4)))
print({k: claims[k] for k in ("username", "client_id", "token_use", "exp") if k in claims})

## 2. El agente obtiene la identidad del token

In [ ]:
%%writefile NovaAssist/app/NovaAgent/identidad.py
"""Identidad del cliente a partir del JWT que el Runtime YA ha validado (firma, emisor, caducidad)."""
import base64
import json


def claims_del_token(auth_header: str) -> dict:
    token = auth_header.split(" ", 1)[1]
    payload = token.split(".")[1]
    payload += "=" * (-len(payload) % 4)
    return json.loads(base64.urlsafe_b64decode(payload))


def actor_desde_contexto(context) -> str | None:
    """Devuelve el username del token o None. Nunca usa datos del payload, que el cliente puede falsificar."""
    headers = getattr(context, "request_headers", None) or {}
    auth = headers.get("Authorization") or headers.get("authorization")
    if not auth or not auth.lower().startswith("bearer "):
        return None
    return claims_del_token(auth).get("username")

In [ ]:
%%writefile NovaAssist/app/NovaAgent/main.py
from bedrock_agentcore.runtime import BedrockAgentCoreApp
from strands import Agent
from strands.agent.conversation_manager import SlidingWindowConversationManager

from contexto import VENTANA_MENSAJES, system_prompt
from gateway import cliente_gateway, token_gateway
from identidad import actor_desde_contexto
from memoria import crear_session_manager, retrieval_config
from modelo import crear_modelo
from respuesta import respuesta

app = BedrockAgentCoreApp()


@app.entrypoint
def invoke(payload, context):
    actor_id = actor_desde_contexto(context)   # identidad verificada; payload["cliente_id"] se IGNORA
    if not actor_id:
        return {"respuesta": "No se ha podido identificar al cliente.", "herramientas": [], "tokens": {}}
    gateway = cliente_gateway(token_gateway())
    with gateway, crear_session_manager(actor_id, context.session_id, retrieval_config()) as sm:
        agent = Agent(
            model=crear_modelo(),
            system_prompt=system_prompt(payload),
            tools=gateway.list_tools_sync(),
            session_manager=sm,
            conversation_manager=SlidingWindowConversationManager(window_size=VENTANA_MENSAJES),
            callback_handler=None,
        )
        result = agent(payload.get("prompt", ""))
    return respuesta(result, actor=actor_id)


if __name__ == "__main__":
    app.run()

Prueba local: el token es de **C-001**, pero el payload dice **C-002**. El agente debe ignorar el payload.

> ✋ **Predice antes de ejecutar.** Token de C-001 y payload que dice C-002: ¿a quién atenderá el agente? ¿Y si no hay token?

In [ ]:
os.environ["NOVA_GATEWAY_TOKEN"] = token_m2m()
probar_local("¿Qué pedidos tengo y cómo preferís contactarme?", cliente_id="C-002",
             headers={"Authorization": f"Bearer {token_cliente('C-001')}"})
probar_local("Hola", headers={})   # sin token → no identificado
del os.environ["NOVA_GATEWAY_TOKEN"]

## 3. Autorizador JWT del Runtime y cabecera `Authorization`

Dos cambios en la entrada del runtime de `agentcore.json`:

1. **Autorizador JWT** (`authorizerType: CUSTOM_JWT`) con el *discovery URL* y el *client id* del pool de clientes: el Runtime valida firma, emisor, caducidad y cliente de cada token **antes** de ejecutar el agente.
2. **Allowlist de la cabecera `Authorization`** (`requestHeaderAllowlist`), para que el código reciba el token ya validado y saque de él el cliente.

Se declara en `agentcore.json` y no en la consola: un `agentcore deploy` posterior revertiría lo que cambies a mano.

In [ ]:
data = agentcore_json()
entrada = next(rt for rt in data["runtimes"] if rt["name"] == "NovaAgent")
entrada["authorizerType"] = "CUSTOM_JWT"
entrada["authorizerConfiguration"] = {"customJwtAuthorizer": {
    "discoveryUrl": state_get("CLIENTES_DISCOVERY_URL"),
    "allowedClients": [state_get("CLIENTES_CLIENT_ID")],
}}
entrada["requestHeaderAllowlist"] = ["Authorization"]
agentcore_json_write(data)
print(json.dumps(entrada, indent=2, ensure_ascii=False))

In [ ]:
assert cli("agentcore validate") == 0, "agentcore.json no es válido: revisa el mensaje de la CLI"
cli("agentcore deploy --yes")

In [ ]:
# Lo que ha quedado configurado en el servicio
rt = boto3.client("bedrock-agentcore-control", region_name=region()).get_agent_runtime(agentRuntimeId=runtime_id())
print(json.dumps({k: rt.get(k) for k in ("authorizerConfiguration", "requestHeaderConfiguration")}, indent=2, default=str))

<details><summary>Si tu versión de la CLI no admitiera estos campos</summary>

El mismo cambio con la API (`update_agent_runtime` reemplaza la configuración entera: hay que reenviar artefacto, rol y red tal como están):

```python
ctl = boto3.client("bedrock-agentcore-control", region_name=region())
actual = ctl.get_agent_runtime(agentRuntimeId=runtime_id())
ctl.update_agent_runtime(
    agentRuntimeId=runtime_id(),
    agentRuntimeArtifact=actual["agentRuntimeArtifact"],
    roleArn=actual["roleArn"],
    networkConfiguration=actual["networkConfiguration"],
    environmentVariables=actual.get("environmentVariables", {}),
    authorizerConfiguration={"customJWTAuthorizer": {
        "discoveryUrl": state_get("CLIENTES_DISCOVERY_URL"),
        "allowedClients": [state_get("CLIENTES_CLIENT_ID")]}},
    requestHeaderConfiguration={"requestHeaderAllowlist": ["Authorization"]},
)
```
Si lo haces así, el siguiente `agentcore deploy` lo revertirá: tendrás que llevarlo también a `agentcore.json`.
</details>

## 4. Verificación

In [ ]:
try:
    invocar("Hola")   # SigV4, sin token de cliente
    D7_CERRADA = False
    print("⚠ Se ha aceptado una invocación sin token: revisa el autorizador")
except Exception as e:  # noqa: BLE001
    D7_CERRADA = True
    print("Sin token →", type(e).__name__, str(e)[:200])

> ✋ **Predice antes de ejecutar.** El token es de **C-001** y el payload dice **C-002**. ¿A quién atenderá el agente? Fíjate en la línea 🪪.

In [ ]:
TOK_C1, TOK_C2 = token_cliente("C-001"), token_cliente("C-002")
invocar("¿Cómo preferís contactarme si hay un problema?", token=TOK_C1, cliente_id="C-002")
actor_1 = ultimo_dato("actor")
invocar("¿Cómo preferís contactarme si hay un problema?", token=TOK_C2)
actor_2 = ultimo_dato("actor")
D4_CERRADA = actor_1 == "C-001" and actor_2 == "C-002"
print("Token C-001 + payload C-002 → atiende a", actor_1, "· token C-002 → atiende a", actor_2)

## 5. Cerrar el acceso indebido (RNF-01): el patrón correcto

El Gateway recibe el token **M2M del agente**: sabe *qué agente* llama, pero no *en nombre de qué cliente*. Para que el sistema de pedidos filtre por propietario:

- **Producción:** usa ***token exchange* On-Behalf-Of (OBO)** de AgentCore Identity, que obtiene para el Gateway un token con la identidad del cliente. Así Policy y la Lambda ven al cliente real.
- **Mínimo:** propaga la identidad por un canal que el modelo no controle y comprueba en la Lambda `pedido.cliente_id == cliente_autenticado`.

**Regla de oro:** nunca confíes en un identificador que el modelo pueda escribir como argumento de una herramienta.

**Ejercicio:** repite la prueba del capítulo 2 (`TN-1002` con el token de C-001) y explica por qué sigue funcionando hasta que implementes OBO.

## Para pensar

- ¿Qué diferencia hay entre autenticar al agente (M2M) y autorizar al usuario (OBO)?
- ¿Por qué decodificar el JWT sin verificar la firma solo es aceptable aquí?

## 🔎 Lo que acabamos de descubrir

Ya sabemos quién es Laura… y aun así **nada impide** que el agente le reembolse 450 € (D2), ni que obedezca a la «autorización» que alguien dejó en la memoria (D5) o a un documento envenenado (D8).

In [ ]:
deuda("D4", "pagada" if D4_CERRADA else "parcial", 9,
      "el cliente sale del JWT validado; el payload se ignora" if D4_CERRADA else "revisa la sección 4: el actor no coincide con el token")
deuda("D7", "pagada" if D7_CERRADA else "parcial", 9,
      "el Runtime rechaza invocaciones sin token de cliente" if D7_CERRADA else "el Runtime aceptó una invocación sin token")
deuda("D1", "parcial", 9, "identidad verificada en el agente; el Gateway todavía no sabe en nombre de qué cliente se llama (OBO)")
cuaderno()
taximetro_cerrar(9)
guardar_partida(9)

📚 **Para repasar más adelante:** la teoría de este capítulo está en [`teoria/cap09-identidad.html`](https://farithjhg.github.io/aws-bedrock-agentcore-course/v7_aprender_haciendo/labs/teoria/cap09-identidad.html) y el mapa completo en [`teoria/index.html`](https://farithjhg.github.io/aws-bedrock-agentcore-course/v7_aprender_haciendo/labs/teoria/index.html). La referencia por componente es la Parte I del documento.

---
➡️ **Siguiente capítulo:** `10_policy_guardrails.ipynb` — *Reglas que el modelo no puede saltarse*

---
**Siguiente:** ➡️ [Capítulo 10 · Reglas que el modelo no puede saltarse](https://colab.research.google.com/github/farithjhg/aws-bedrock-agentcore-course/blob/main/v7_aprender_haciendo/labs/10_policy_guardrails.ipynb) · 🎬 [su escena en la presentación](https://farithjhg.github.io/aws-bedrock-agentcore-course/v7_aprender_haciendo/presentacion_nova_assist.html#cap10). En Colab, tu estado ya está en Drive: el capítulo siguiente lo recupera solo.